<a href="https://colab.research.google.com/github/Biegle-W/google_colab_llm/blob/main/notebooks/diffusers/diffusers_hub_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Colab Diffusers hub

One authenticated OpenAI-compatible API for **Diffusers** models on a **Google AI Pro / Colab Pro** GPU.
**This notebook installs only `diffusers`** and advertises only that engine's catalog ids.

- Supported models: Text-to-image: Qwen-Image-2.1, FLUX, SDXL (+ MiniMax-H3 prefetch)
- Sibling notebooks: `notebooks/llama/`, `notebooks/vllm/`, `notebooks/sglang/`, `notebooks/diffusers/`, `notebooks/strata/`, `notebooks/maya/`, `notebooks/tensorfold/`, `notebooks/comfyui/`, `notebooks/sparsa/` — see `notebooks/llm_hub_colab.ipynb`
- Drive root: `MyDrive/llm_models/` (shared weights/builds; settings file is `config/settings_diffusers.json`)
- Engine matrix: `colab_llm_hub/ENGINE_MATRIX.md`

Runtime → Change runtime type → **L4, A100 or G4**, then run the cells in order.


## 1. Mount Drive, create the workspace, fetch (or restore) the hub code

Each run fetches the latest hub code from GitHub and refreshes its copy on Drive; only if GitHub cannot be reached does it fall back to that Drive copy. Settings and tool binaries are restored from Drive.


In [ ]:
import os, re, shutil, subprocess, sys
from google.colab import drive
drive.mount("/content/drive")

ENGINE   = "diffusers"  # locked for this notebook
HUB_REPO  = "https://github.com/Biegle-W/google_colab_llm"
HUB_REF   = "main"  # always clone hub package from main
DRIVE_DIR = "/content/drive/MyDrive/llm_models"
HUB_DIR   = "/content/hub_repo"

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(f"{cmd}\n{r.stdout}{r.stderr}")
    return r.stdout.strip()

OFFLINE_HUB = False   # True = skip GitHub and use the hub code saved on Drive

os.makedirs(DRIVE_DIR, exist_ok=True)
sys.path.insert(0, HUB_DIR)
ref_key = (HUB_REF or "default").replace("/", "_")
cached = f"{DRIVE_DIR}/builds/hub_pkg/{ref_key}/colab_llm_hub"
have_cache = os.path.isdir(cached) and os.path.isfile(f"{DRIVE_DIR}/builds/hub_pkg/{ref_key}/COMPLETE")
fresh = False
if not OFFLINE_HUB:
    try:                          # always prefer the latest code: fixes land without reopening
        sh(f"rm -rf {HUB_DIR}")
        if HUB_REF and re.fullmatch(r"[0-9a-f]{40}", HUB_REF):
            sh(f"mkdir -p {HUB_DIR} && cd {HUB_DIR} && git init -q && git fetch -q --depth 1 {HUB_REPO} {HUB_REF} && git checkout -q FETCH_HEAD")
        else:
            sh(f"git clone -q --depth 1 {'--branch ' + HUB_REF if HUB_REF else ''} {HUB_REPO} {HUB_DIR}")
        fresh = True
        print("Fetched hub code from GitHub:", sh(f"cd {HUB_DIR} && git rev-parse --short HEAD"))
    except RuntimeError as e:
        print("Could not fetch from GitHub; using the copy on Drive.", str(e)[-300:])
if not fresh:
    if not have_cache:
        raise RuntimeError("No hub code: GitHub was unreachable and there is no copy on Drive yet.")
    shutil.rmtree(HUB_DIR, ignore_errors=True)
    os.makedirs(HUB_DIR, exist_ok=True)
    shutil.copytree(cached, f"{HUB_DIR}/colab_llm_hub")
    print("Restored hub package from Drive (it may be older than GitHub)")

from colab_llm_hub.workspace import open_workspace
from colab_llm_hub import install
WS = open_workspace(DRIVE_DIR, engine=ENGINE)
print("Drive workspace:", WS.describe())
if fresh:                         # keep the Drive copy current for offline fallback
    install.cache_hub_package(HUB_DIR, str(WS.builds), ref=HUB_REF or "default")
import colab_llm_hub
print("hub", colab_llm_hub.__version__, "engine", ENGINE)


## 2. Detect GPU, CPU, and RAM — then list this engine's models


In [ ]:
from colab_llm_hub import catalog, gpu, recommend

G = gpu.detect()[0]
HOST = gpu.print_runtime(G)  # GPU + CPU + RAM + disk, every run
CPU, RAM = HOST["cpu"], HOST["ram"]
VRAM = G["vram_gb"]
ALL = catalog.load()
ENTRIES = catalog.for_engines(ALL, [ENGINE])
PLAN = recommend.for_engine(ENGINE, VRAM, ALL)
print()
print(recommend.explain(VRAM, ALL, engine=ENGINE))
print()
print(f"{'model id':36s} {'kind':5s} {'size':>7s}  {'ctx/edge':>14s}")
for e in ENTRIES:
    fit = catalog.fits(e, VRAM)
    kind = catalog.kind_for(e)
    label = "edge" if kind in ("image", "video") else "ctx"
    ctx = f"{catalog.context_for(e, VRAM):,} {label}" if fit else f"needs {catalog.min_vram_gb(e)} GB"
    mark = " *" if e["id"] in PLAN["prefetch"] else ""
    print(f"{e['id']:36s} {kind:5s} {e['size_gb']:6.1f}G  {ctx:>14s}{mark}" + ("" if fit else "   (does not fit)"))
print("\n* = recommended prefetch for this GPU + engine")


## 3. Config (saved on Drive under `config/settings_diffusers.json`)

Defaults come from the engine recommender, then from any settings already on Drive. Edit and re-run to persist.


In [ ]:
from colab_llm_hub import catalog, gpu, recommend

G = gpu.detect()[0]
HOST = gpu.print_runtime(G)  # re-detect GPU + CPU + RAM every config run
CPU, RAM = HOST["cpu"], HOST["ram"]
VRAM = G["vram_gb"]
ALL = catalog.load()
ENTRIES = catalog.for_engines(ALL, [ENGINE])
PLAN = recommend.for_engine(ENGINE, VRAM, ALL)
saved = WS.load_settings()

PREFETCH = saved.get("prefetch") or PLAN["prefetch"]
ENGINES = [ENGINE]  # never install other engines from this notebook
CTX_OVERRIDE = saved.get("ctx_override") or {}
LLAMA_CPP_REF = saved.get("llama_cpp_ref") or "4c3c5810f87efe177949f1b85ae509d38a2a311f"
REBUILD_LLAMA = bool(saved.get("rebuild_llama", False))
REFRESH_WHEELS = bool(saved.get("refresh_wheels", False))
# Default: Tailscale. Auth key is stored on private Drive (hub/tailscale_authkey), never in git.
# Alternatives: set TUNNEL to "quick" or "named" and keep_cloudflare_quick=True for Cloudflare.
TUNNEL = saved.get("tunnel") or "tailscale"
if TUNNEL == "quick" and not saved.get("keep_cloudflare_quick"):
    TUNNEL = "tailscale"  # migrate older Drive defaults to Tailscale
PORT = int(saved.get("port") or 8000)
TS_HOSTNAME = saved.get("ts_hostname") or f"colab-{ENGINE}"
TS_FUNNEL = bool(saved.get("ts_funnel", False))  # False = tailnet-only Serve
REFRESH_TS_KEY = bool(saved.get("refresh_tailscale_key", False))

if not PREFETCH:
    # Fall back to the smallest model that fits so the notebook is still runnable.
    fit_ids = [e["id"] for e in ENTRIES if catalog.fits(e, VRAM)]
    PREFETCH = fit_ids[:1]
    print("No band default; using", PREFETCH)

WS.save_settings({
    "engine": ENGINE, "prefetch": PREFETCH, "engines": ENGINES,
    "ctx_override": CTX_OVERRIDE, "llama_cpp_ref": LLAMA_CPP_REF,
    "rebuild_llama": REBUILD_LLAMA, "refresh_wheels": REFRESH_WHEELS,
    "tunnel": TUNNEL, "port": PORT, "hub_ref": HUB_REF,
    "ts_hostname": TS_HOSTNAME, "ts_funnel": TS_FUNNEL,
    "refresh_tailscale_key": False,  # set True once to re-prompt and overwrite Drive key
    "gpu_name": G["name"], "vram_gb": VRAM, "band": PLAN["band"],
    "cpu_model": CPU["model"], "cpu_cores": CPU["cores"], "cpu_threads": CPU["threads"],
    "ram_gb": RAM["total_gb"], "ram_available_gb": RAM["available_gb"],
})
print("Saved settings →", WS.settings_path)
print("PREFETCH =", PREFETCH)
print("ENGINE   =", ENGINE)
print("TUNNEL   =", TUNNEL)
print("Tailscale key file (private Drive, not git):", WS.hub / "tailscale_authkey")
unknown = [m for m in PREFETCH if m not in {e["id"] for e in ENTRIES}]
assert not unknown, f"unknown or wrong-engine model ids: {unknown}"
not_fit = [m for m in PREFETCH if not catalog.fits(catalog.by_id(ENTRIES)[m], VRAM)]
assert not not_fit, f"these do not fit this GPU: {not_fit}"
assert all(catalog.by_id(ENTRIES)[m]["backend"] == ENGINE for m in PREFETCH)


## 4. Install Diffusers (built or downloaded once, then restored from Drive)


In [ ]:
from colab_llm_hub import install

BUILDS = str(WS.builds)
LLAMA_BIN = "llama-server"
STRATA_BIN = "strata"
MAYA_ROOT = str(WS.builds / "project-maya")
TENSORFOLD_BIN = "tensorfold-native"
COMFYUI_ROOT = str(WS.builds / "ComfyUI")
SPARSA_ROOT = str(WS.builds / "Sparsa")
install.ensure_hub_deps(BUILDS, refresh=REFRESH_WHEELS)
if ENGINE == "llama":
    LLAMA_BIN = install.ensure_llama(BUILDS, G["arch"], ref=LLAMA_CPP_REF, rebuild=REBUILD_LLAMA)
    print("llama-server:", LLAMA_BIN)
elif ENGINE == "vllm":
    print("vLLM", install.ensure_vllm(BUILDS, refresh=REFRESH_WHEELS))
elif ENGINE == "sglang":
    print("SGLang", install.ensure_sglang(BUILDS, refresh=REFRESH_WHEELS))
elif ENGINE == "diffusers":
    print("diffusers", install.ensure_diffusers(BUILDS, refresh=REFRESH_WHEELS))
elif ENGINE == "strata":
    STRATA_BIN = install.ensure_strata(BUILDS, refresh=REFRESH_WHEELS)
    print("strata:", STRATA_BIN)
elif ENGINE == "maya":
    MAYA_ROOT = install.ensure_maya(BUILDS, refresh=REFRESH_WHEELS)
    print("maya:", MAYA_ROOT)
elif ENGINE == "tensorfold":
    TENSORFOLD_BIN = install.ensure_tensorfold(BUILDS, refresh=REFRESH_WHEELS)
    print("tensorfold:", TENSORFOLD_BIN)
elif ENGINE == "comfyui":
    COMFYUI_ROOT = install.ensure_comfyui(BUILDS, refresh=REFRESH_WHEELS)
    print("comfyui:", COMFYUI_ROOT)
elif ENGINE == "sparsa":
    SPARSA_ROOT = install.ensure_sparsa(BUILDS, refresh=REFRESH_WHEELS)
    print("sparsa:", SPARSA_ROOT)
print("Install log →", WS.logs / "install.log")


## 5. Download the models to Google Drive (skipped for anything already saved)


In [ ]:
import threading, time
from colab_llm_hub.store import ModelStore

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

store = ModelStore(DRIVE_DIR, str(WS.local_models))
for mid in PREFETCH:
    e = catalog.by_id(ENTRIES)[mid]
    if store.state(e)["on_drive"]:
        print(f"{mid}: already on Drive")
        continue
    stop = threading.Event()
    def report(mid=mid, e=e):
        while not stop.wait(15):
            p = store.progress.get(mid, {})
            print(f"  {mid}: {p.get('phase', '')} {p.get('done_bytes', 0) / 1e9:.1f} / {e['size_gb']:.1f} GB")
    threading.Thread(target=report, daemon=True).start()
    t0 = time.time()
    try:
        store.prefetch(e)
    finally:
        stop.set()
    print(f"{mid}: saved to Drive in {(time.time() - t0) / 60:.1f} min")


## 6. Start the hub and open Tailscale

Re-running this cell restarts the hub. Secrets stay on **your private Google Drive** (`MyDrive/llm_models/hub/`), never in this public GitHub repo or in settings JSON:

- `hub/api_key` — hub Bearer token (auto-created)
- `hub/tailscale_authkey` — Tailscale auth key (prompted once, then reused)

Create a **reusable + ephemeral** key at [Tailscale Keys](https://login.tailscale.com/admin/settings/keys). Set `REFRESH_TS_KEY = True` (or `refresh_tailscale_key` in settings) to overwrite the Drive copy. Cloudflare: set `TUNNEL="quick"` / `"named"` with `keep_cloudflare_quick=True`.


In [ ]:
import json, requests, time
from colab_llm_hub import keyfile, tunnel, provider

subprocess.run("pkill -TERM -f '[c]olab_llm_hub serve'; pkill -f '[c]loudflared'; pkill -TERM -f '[t]ailscaled.*colab-llm-tailscale'", shell=True)
time.sleep(3)

API_KEY = keyfile.load_or_create(str(WS.hub / "api_key"))
HUB_LOG = str(WS.logs / f"hub_{ENGINE}.log")
args = [sys.executable, "-m", "colab_llm_hub", "serve", "--drive-dir", DRIVE_DIR,
        "--local-dir", str(WS.local_models), "--log-dir", str(WS.engine_logs),
        "--port", str(PORT), "--llama-bin", LLAMA_BIN,
        "--strata-bin", STRATA_BIN, "--maya-root", MAYA_ROOT,
        "--tensorfold-bin", TENSORFOLD_BIN, "--comfyui-root", COMFYUI_ROOT,
        "--sparsa-root", SPARSA_ROOT,
        "--engines", ENGINE]
for k, v in CTX_OVERRIDE.items():
    args += ["--ctx", f"{k}={v}"]
hub_proc = subprocess.Popen(args, cwd=HUB_DIR, env=dict(os.environ, PYTHONPATH=HUB_DIR, COLAB_LLM_DRIVE=DRIVE_DIR),
                            stdout=open(HUB_LOG, "w"), stderr=subprocess.STDOUT)

AUTH = {"Authorization": f"Bearer {API_KEY}"}
for _ in range(60):
    try:
        if requests.get(f"http://127.0.0.1:{PORT}/health", headers=AUTH, timeout=2).status_code == 200:
            break
    except requests.RequestException:
        pass
    if hub_proc.poll() is not None:
        raise RuntimeError("The hub exited:\n" + open(HUB_LOG).read()[-3000:])
    time.sleep(1)
else:
    raise RuntimeError("The hub did not start:\n" + open(HUB_LOG).read()[-3000:])

if TUNNEL == "tailscale":
    # Load from private Drive, or prompt once and save there (never committed to git).
    TS_KEY_PATH = str(WS.hub / "tailscale_authkey")
    TS_AUTHKEY = tunnel.load_or_ask_tailscale_authkey(
        TS_KEY_PATH, refresh=REFRESH_TS_KEY)
    tunnel_proc, BASE = tunnel.start_tailscale(
        PORT, TS_AUTHKEY, hostname=TS_HOSTNAME, drive_dir=DRIVE_DIR, funnel=TS_FUNNEL)
    del TS_AUTHKEY  # drop from notebook namespace; Drive file remains private
elif TUNNEL == "named":
    from google.colab import userdata
    tunnel.ensure_cloudflared(drive_dir=DRIVE_DIR)
    tunnel_proc = tunnel.start_named(userdata.get("CLOUDFLARE_TUNNEL_TOKEN"), drive_dir=DRIVE_DIR)
    BASE = "https://" + userdata.get("CLOUDFLARE_HOSTNAME").removeprefix("https://").strip("/")
else:
    tunnel.ensure_cloudflared(drive_dir=DRIVE_DIR)
    tunnel_proc, BASE = tunnel.start_quick(PORT, drive_dir=DRIVE_DIR)

MODELS = requests.get(f"http://127.0.0.1:{PORT}/v1/models", headers=AUTH).json()["data"]
WS.save_endpoint(BASE + "/v1", API_KEY, G["name"], [m["id"] for m in MODELS])

print("Engine   :", ENGINE)
print("Tunnel   :", TUNNEL)
print("Base URL :", BASE + "/v1")
print("API key  :", API_KEY)
print(f"Models   : {', '.join(m['id'] for m in MODELS)}")
print("Endpoint →", WS.hub / "endpoint.json")
if TUNNEL == "tailscale" and not TS_FUNNEL:
    print("Reach this URL from any device on your Tailscale tailnet.")
print()
print("Any OpenAI-compatible client:")
print(provider.openai_env(BASE, API_KEY))
print("dsh: on your computer, in a clone of this repo:")
print(f"  python -m colab_llm_hub.configure_dsh --url {BASE} --key {API_KEY} --settings <path to dsh settings.yaml>")
print(f"  export COLAB_LLM_API_KEY={API_KEY}\n")
print(provider.dsh_provider_yaml(BASE, MODELS))


## 7. Try image generation

The first request loads the pipeline (Drive→local copy, then GPU). Non-streaming may return `503` + `Retry-After` while loading — re-run the cell.


In [ ]:
from IPython.display import display, Image as IPyImage
import base64

by = catalog.by_id(ENTRIES)
image_models = [m for m in MODELS if m["id"] in by and catalog.kind_for(by[m["id"]]) == "image"]
if not image_models:
    raise RuntimeError("No image models visible for this GPU; lower size or pick another Diffusers id in PREFETCH.")

IMG = next((m for m in PREFETCH if m in {x["id"] for x in image_models}), image_models[0]["id"])
prompt = "a watercolor fox under pine trees, soft light"
t0 = time.time()
r = requests.post(f"http://127.0.0.1:{PORT}/v1/images/generations", headers=AUTH, timeout=900,
                  json={"model": IMG, "prompt": prompt, "size": "768x768", "n": 1})
if r.status_code == 503 and r.json().get("error", {}).get("code") == "model_loading":
    print("Image model still loading; re-run this cell in a minute.")
else:
    j = r.json()
    print("=== image:", IMG, j.get("size"), "===")
    if r.ok:
        display(IPyImage(data=base64.b64decode(j["data"][0]["b64_json"])))
        path = WS.save_image_b64(j["data"][0]["b64_json"], IMG, prompt=prompt)
        print(f"[{time.time() - t0:.1f} s] Saved →", path)
    else:
        print(j)


## 8. Status and logs (re-run any time)


In [ ]:
st = requests.get(f"http://127.0.0.1:{PORT}/v1/hub/status", headers=AUTH).json()
WS.save_json(WS.status, f"hub-status-{ENGINE}", st)
print(json.dumps({k: st[k] for k in ("gpu", "vram_gb", "active", "loading", "phase", "inflight", "errors", "local_free_gb")}, indent=1))
for mid, p in st["downloads"].items():
    print(f"  {mid}: {p.get('phase')} {p.get('done_bytes', 0) / 1e9:.1f} / {p.get('total_bytes', 0) / 1e9:.1f} GB")
print("\n--- hub log (Drive) ---"); print(open(HUB_LOG).read()[-1500:])
!nvidia-smi --query-gpu=memory.used,memory.total --format=csv
print("Outputs →", WS.outputs)


## 9. Stop


In [ ]:
hub_proc.terminate()        # stops the hub and the engine it started
try:
    tunnel_proc.terminate()
except Exception:
    pass
if TUNNEL == "tailscale":
    tunnel.stop_tailscale(drive_dir=DRIVE_DIR)


## Notes

### This notebook = `diffusers` only

| Catalog id | Notes |
|---|---|
| `qwen-image-2.1-nf4` | ≤24 GB |
| `qwen-image-2.1` | BF16 on A100/G4 |
| `flux-schnell`, `sdxl-*`, `sd15` | Other Diffusers entries |
| `minimax-h3` | Prefetch only (video); prefer **ComfyUI** notebook |

### Remote access (Tailscale by default)

1. Create a **reusable + ephemeral** auth key: https://login.tailscale.com/admin/settings/keys
2. First run: cell 6 prompts for it and saves to **private Drive** `MyDrive/llm_models/hub/tailscale_authkey` (not GitHub).
3. Later Colab sessions reuse that Drive file automatically.
4. Optional Colab secret `TAILSCALE_AUTHKEY` if you press Enter on an empty prompt.
5. Connect from your PC (same tailnet) to the printed `https://….ts.net/v1` URL.
6. Cloudflare fallbacks: `TUNNEL="quick"` / `"named"` with `keep_cloudflare_quick=True`.

### Drive layout

| Path under `MyDrive/llm_models` | Contents |
|---|---|
| `models/<id>/` | Weights shared across engine notebooks |
| `builds/` | llama.cpp tarballs, wheels, hub_pkg, cloudflared, Tailscale |
| `config/settings_diffusers.json` | Prefetch / ctx / tunnel for **this** engine (no secrets) |
| `hub/api_key` | Hub Bearer token (private Drive only) |
| `hub/tailscale_authkey` | Tailscale auth key (private Drive only; never in git) |
| `hub/endpoint.json` | Last base URL + key hint |
| `hf_cache/`, `logs/`, `outputs/` | Cache, logs, generation results |

- See `colab_llm_hub/ENGINE_MATRIX.md` for workstation (Maya / Strata / HyperQwen) notes.
- Set `REFRESH_WHEELS` / `REBUILD_LLAMA` / `refresh_tailscale_key` in settings to force updates.
- **Security:** Tailscale Serve is tailnet-only. Cloudflare quick tunnels are public; the hub API key still protects GPU use. Never commit `hub/api_key` or `hub/tailscale_authkey`.
